# 16 — LangGraph: the graph archetype

**What you'll learn**

- That a LangGraph agent is an explicit graph of nodes and edges over a first-class `TypedDict` state — the `run_agent` loop (ch02) and the `route` router (ch05) you hand-built, now declarative
- The quick path: `create_react_agent(model, tools)` triaging a Larkspur ticket, and that it compiles to a `CompiledStateGraph` — a graph under the hood, not a while-loop
- The hand-built path: a `StateGraph` with `triage` / `tools` / `issue_refund` nodes and an `add_conditional_edges` router that dispatches exactly like your ch05 `route()`
- `interrupt_before` plus a `MemorySaver` checkpointer as the ch08 approval gate made declarative — run until it pauses, inspect the state, resume with `graph.invoke(None, config)`
- The honest trade: graph boilerplate versus the persistence, human-in-the-loop, and streaming you get for free

*Time: ~2 min. Cost: ~$0.03. LangChain's client talks to OpenRouter directly, so these calls skip the LiteLLM disk cache — a rerun costs the same cent or two.*

> **Before running this notebook:** `pip install -e ".[langgraph]"` (once). It pulls in `langgraph` and `langchain-openai` — the graph runtime and the chat-model client we point at OpenRouter. Everything else stays the same.

## Every agent you built was already a graph

Chapter 02's `run_agent` is a while-loop: call the model, run whatever tools it asked for, feed the results back, repeat until it finishes. Chapter 05 put a `route()` in front of it — one cheap call that sorts a ticket into a lane. Chapter 08 wrapped the risky tools in `require_approval` and taught the run to checkpoint its state so a crash could resume. Each was a piece of plumbing you can read line by line.

[LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) is those same pieces, drawn as a graph. Its docs call it a "low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents." The loop becomes **nodes** joined by **edges**; the message list becomes **first-class state** with a reducer; the router becomes a **conditional edge**; the approval gate becomes an **interrupt**; and the checkpoint becomes a **checkpointer** you attach at compile time. Nothing here is a new idea. What LangGraph gives you is that the ideas are now declarative — you describe the graph, and the runtime handles stepping through it, persisting it, pausing it, and resuming it.

This chapter walks both doors into that runtime. First the prebuilt one-liner, then the same triage built node by node, so the graph stops being a black box and becomes the machinery you already own — rearranged.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

A LangGraph run is a sequence of node executions, each with its own model and tool calls. Phoenix traces them under LangChain's own instrumentation, so you can watch the graph step: the `triage` node, the loop back through `tools`, the pause at the gate, the resume. Optional as always — skip it and nothing else changes. Note the seam has moved: these calls go through LangChain's client, not `shoplab.llm.complete`, so the cost `LEDGER` from earlier chapters no longer sees them.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The model: one chat client, pointed at OpenRouter

LangGraph nodes call the model through a LangChain chat model rather than `shoplab.llm.complete`. It is the same wiring chapter 11 used for `deepagents`: a `ChatOpenAI` aimed at OpenRouter's OpenAI-compatible endpoint. One catch worth repeating — the model id drops the `openrouter/` prefix. That prefix is LiteLLM's routing syntax; the raw OpenAI API just wants `deepseek/deepseek-v3.2`.

In [ ]:
import os
from langchain_openai import ChatOpenAI
from shoplab import world

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}

model = ChatOpenAI(base_url="https://openrouter.ai/api/v1",
                   api_key=os.environ["OPENROUTER_API_KEY"],
                   model="deepseek/deepseek-v3.2", temperature=TEMPERATURE)
print("model is a:", type(model).__module__ + "." + type(model).__name__)

## The tools you already have, as LangChain callables

The graph needs the same Larkspur lookups the ops desk always uses. LangChain's `@tool` decorator turns a plain annotated function into a tool the model can call — the docstring becomes the description, the type hints become the JSON schema. This is exactly what `shoplab.tools.to_openai_tools` did by hand in chapter 02; here the decorator does the wiring. We expose the three read-only lookups (the risky `issue_refund` becomes a *node* below, not a tool, so the graph can gate it).

In [ ]:
from langchain_core.tools import tool

@tool
def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

@tool
def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history)."""
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

@tool
def find_policy(query: str, k: int = 2) -> list:
    """Keyword-search the 12 Larkspur store policy documents."""
    return world.search_policy(query, k=k)

read_tools = [get_order, get_customer, find_policy]
print("read tools:", [t.name for t in read_tools])

## The quick path: a prebuilt graph

The fastest way in is `create_react_agent`: hand it a model and a list of tools and it returns a ready-to-run agent. Under the hood it is the ReAct loop of chapter 02 — reason, call a tool, observe, repeat — assembled for you. We give it the ops-desk brief and the fixed appendix ticket `TKT-2205` (an opened, in-window return from a non-vip member; gold is `partial_refund` under `pol-restocking` for $170.99) and read the answer off the final message.

Print the compiled object's type and note what it is: not an agent class, but a `CompiledStateGraph`. The quick path and the hand-built path below produce *the same kind of thing* — a graph.

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM = ("You are the Larkspur ops desk. Look up the order and customer, find the "
          "governing policy, then end with a line exactly like "
          "'DECISION: partial_refund | pol-restocking | 170.99' using one of "
          "approve_refund/partial_refund/replacement/store_credit/deny/escalate. Be terse.")
TICKET = ("Triage Larkspur ticket TKT-2205 (order ORD-7312, customer CUST-07, sku LK-1016, "
          "qty 1): 'I opened the box and used the Torrent boots one evening indoors, they "
          "pinch at the toes. Repacked with tags. Refund my original payment method.'")

agent = create_react_agent(model, read_tools, prompt=SYSTEM)
print("compiled type:", type(agent).__module__ + "." + type(agent).__name__)
out = agent.invoke({"messages": [{"role": "user", "content": TICKET}]})
print("messages in final state:", len(out["messages"]))
print("final answer:\n" + out["messages"][-1].content[-320:])

> **What you should see:** `create_react_agent` returns a `langgraph.graph.state.CompiledStateGraph`, and running it lands on the right call — the member (not vip) opened return names `pol-restocking`, applies the 10% restocking fee, and arrives at $170.99 ($189.99 x 0.90). A handful of messages accumulate in `out["messages"]` as it reads the order, the customer, and the policy. (You will also see a one-line deprecation notice: in LangGraph 1.x the prebuilt is migrating to `langchain.agents.create_agent`. The framework churns; the graph it builds does not.)

## The graph, by hand: state and nodes

The prebuilt hides the structure. Now build the same triage explicitly, so every seam is visible. Three ingredients.

First, **state**. A LangGraph graph threads a shared state object through its nodes; ours is a `TypedDict` with one channel, `messages`, annotated with the `add_messages` reducer so each node's output is *appended* to the running list rather than replacing it. That list is the same message history chapter 02 threaded through the loop by hand — the reducer is just the append made declarative.

Then, **nodes**. A node is a function `state -> partial-state-update`. `triage` is the reasoning node: it calls the model (bound to the read tools) and returns its message. `issue_refund` is the risky node — the money move — which we keep separate precisely so the graph can stop in front of it.

In [ ]:
import re
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode
from shoplab.tools import Ledger

class TriageState(TypedDict):
    messages: Annotated[list, add_messages]     # reducer: append, don't overwrite

llm_with_tools = model.bind_tools(read_tools)
ledger = Ledger()                               # ch02's real side-effect log

def triage(state: TriageState):                 # the reasoning node == ch02's model call
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def issue_refund(state: TriageState):           # the RISKY node -- writes the ledger
    figs = re.findall(r"\d+\.\d{2}", state["messages"][-1].content)
    entry = ledger.record("issue_refund", order_id="ORD-7312",
                          amount_usd=float(figs[-1]) if figs else 0.0)
    return {"messages": [{"role": "assistant", "content": f"REFUND EXECUTED -> {entry}"}]}

### The conditional edge is your ch05 router

Third ingredient: **edges**, and the interesting one is *conditional*. A conditional edge is a function that reads the state and returns the name of the next node — which is exactly what chapter 05's `route(ticket)` did when it picked a handling lane. Here `route` looks at the last message: if the model asked for a tool, go to the `tools` node; if it proposed a money decision, go to `issue_refund`; otherwise the run is done. Same job as ch05, wired into the graph instead of an `if` in your code.

In [ ]:
def route(state: TriageState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):       # model wants a read tool -> tool node
        return "tools"
    text = (last.content or "").lower()
    if "partial_refund" in text or "approve_refund" in text:  # a money move -> risky node
        return "issue_refund"
    return END

Now assemble the graph: add the three nodes (the read tools go in a `ToolNode`, LangGraph's prebuilt dispatcher — the graph-native `run_tool` from chapter 02), wire `START` to `triage`, attach the conditional edge, and loop `tools` back to `triage`. `build_graph` returns the un-compiled builder so we can compile it two ways — first plain, then gated. Compile it plain, print the edges, and run it once.

In [ ]:
def build_graph():
    g = StateGraph(TriageState)
    g.add_node("triage", triage)
    g.add_node("tools", ToolNode(read_tools))
    g.add_node("issue_refund", issue_refund)
    g.add_edge(START, "triage")
    g.add_conditional_edges("triage", route, ["tools", "issue_refund", END])
    g.add_edge("tools", "triage")               # loop back after each read tool (ch02)
    g.add_edge("issue_refund", END)
    return g

ungated = build_graph().compile()
for e in ungated.get_graph().edges:
    print(f"  {e.source:12} -> {e.target:12}{'  [conditional]' if e.conditional else ''}")

seed = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": TICKET}]
out = ungated.invoke({"messages": seed})
print("\nlast message:", out["messages"][-1].content[:55])
print("ledger entries after ungated run:", len(ledger.entries))   # money moved, no gate

> **What you should see:** the printed edges spell out the graph — `START -> triage`, `tools -> triage`, and three `[conditional]` edges out of `triage` (to `tools`, `issue_refund`, and `__end__`). Running it, the model reads the order, customer, and policy through the `tools` loop, proposes `partial_refund`, and the router sends it to `issue_refund` — which *executes* — writing a real entry to the ch02 `Ledger` (`ledger entries after ungated run: 1`) and printing "REFUND EXECUTED". That is the whole point of the next section: a plain graph is your ch02 loop plus a ch05 router, and it will move money — a side effect you can count, not just a printed claim — because nothing is standing in front of the risky node.

## The interrupt is your approval gate

Chapter 08 wrapped `issue_refund` in `require_approval` so money moved only after a human said yes, and gave the loop a `Checkpoint` so a paused run had state to resume from. LangGraph ships both as first-class features, and they compose into one line.

Attach a `MemorySaver` [checkpointer](https://docs.langchain.com/oss/python/langgraph/persistence) at compile time — its docs describe persistence as giving agents "short-term memory through checkpointers" — and pass `interrupt_before=["issue_refund"]`. Now [invoking the graph](https://docs.langchain.com/oss/python/langgraph/interrupts) runs until it is *about* to enter the risky node and then stops: "interrupts allow you to pause graph execution at specific points and wait for external input before continuing." The proposed decision is sitting in the checkpointed state, waiting for a human. This is `require_approval` (the pause) and `Checkpoint` (the saved state) fused — declarative, and durable across processes because the state lives on the checkpointer, keyed by a `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph = build_graph().compile(checkpointer=checkpointer,
                              interrupt_before=["issue_refund"])   # the ch08 gate

ledger.entries.clear()                          # fresh ledger for the gated demo
config = {"configurable": {"thread_id": "tkt-2205"}}
state = graph.invoke({"messages": seed}, config)
snap = graph.get_state(config)
print("checkpointer:", type(checkpointer).__name__)
print("PAUSED. next node waiting =", snap.next)
print("ledger entries so far:", len(ledger.entries))   # 0 -- the gate stopped the write
print("proposal:", state["messages"][-1].content[-80:])

> **What you should see:** the run halts with `next node waiting = ('issue_refund',)` — it stopped *before* the money move, exactly as `interrupt_before` promises. The model's proposal (ending in a `DECISION:` line for `partial_refund | pol-restocking | 170.99`) is already in the state, checkpointed under `thread_id` `tkt-2205`. The ledger still reads zero (`ledger entries so far: 0`) — nothing was refunded yet; the graph is parked at the gate. The concrete class is `InMemorySaver` (`MemorySaver` is its alias) — swap it for `SqliteSaver` or `PostgresSaver` and the same pause survives a process restart.

In [ ]:
# A human reviews snap.values and approves (ch08's approver returns True).
# Resume by invoking with None: the checkpointer reloads the paused state,
# the gate lifts, and the risky node finally runs.
final = graph.invoke(None, config)
print("AFTER RESUME:", final["messages"][-1].content)
print("ledger entries now:", len(ledger.entries), "(the write finally happened)")
print("next node waiting =", graph.get_state(config).next)

> **What you should see:** resuming with `graph.invoke(None, config)` picks up from the checkpoint — no re-reading the order or re-querying policy — runs the `issue_refund` node — which writes the ledger (`ledger entries now: 1`, up from `0` at the gate) and prints "REFUND EXECUTED". `get_state` now reports `next node waiting = ()`: the graph is done. That is the ch08 story end to end — pause at the gate, inspect, approve, resume — with the pause, the state, and the resume all handed to you by the runtime instead of hand-wired into the loop, and one ledger entry proving exactly one money move happened, only after approval.

## Machinery map: LangGraph feature to the part you built

Line them up and the framework stops being magic. Every LangGraph concept in this chapter maps to a piece of machinery you built by hand in Parts 1-3 — usually a graph feature standing in for a few lines of your own code.

| LangGraph concept | Your hand-built equivalent | Built in |
|---|---|---|
| `StateGraph` nodes + the `triage` -> `tools` -> `triage` loop | `run_agent`'s while-loop: model call -> tool calls -> results, repeat | ch02 |
| `TypedDict` state + the `add_messages` reducer | the `messages` list threaded through the loop; appending to it by hand | ch02 |
| `ToolNode` | `run_tool` dispatching one tool call and never raising | ch02 |
| `add_conditional_edges("triage", route, ...)` | `route(ticket)` choosing a handling lane | ch05 |
| `interrupt_before=["issue_refund"]` | `require_approval` wrapping a risky tool in a yes/no gate | ch08 |
| `MemorySaver` checkpointer + `thread_id` | `Checkpoint.save` / `load` of `{messages, step, meta}` | ch08 |
| `graph.invoke(None, config)` to resume | the `DurableRunner` reloading a checkpoint and continuing | ch12 |
| `create_react_agent(model, tools)` | assembling the loop and toolset yourself | ch02 |

## The honest trade

Read the map left to right and LangGraph is a real gift. You declared three nodes and an edge and got, for free: a checkpointer that persists state across processes, a human-in-the-loop interrupt that pauses exactly where you asked, streaming of intermediate steps, and time-travel over the checkpointed history. Every one of those is something you *could* build — you did build the gate and the checkpoint — but here they arrive assembled, tested, and maintained. For a durable, pausable production agent, that is often the right buy, and nothing you learned is wasted: you can reason about the graph precisely *because* you built each piece once.

Read it right to left and the cost is just as real. The hand-built graph took a state schema, a builder, three `add_node` calls, and explicit edges to express what a fifteen-line while-loop already did — graph ceremony you pay up front. The router that was an `if` in your code is now a function wired into `add_conditional_edges`, a level of indirection you have to hold in your head. And the framework moves: `create_react_agent` is already deprecated toward `langchain.agents.create_agent`, so the convenience layer you lean on today may be renamed tomorrow. The deepest cost is the one chapter 11 flagged — the seam moved. Your calls now run through `ChatOpenAI`, not `shoplab.llm.complete`, so the cost `LEDGER` and your own tracing no longer see them unless you re-instrument. Persistence, interrupts, and streaming are the trade you make for that; know what you are handing over before you sign.

## Recap

| Concept | One-liner |
|---|---|
| Graph, not loop | a LangGraph agent is nodes joined by edges over first-class state — your ch02 loop and ch05 router, made declarative. |
| `create_react_agent` | the prebuilt one-liner; it compiles to a `CompiledStateGraph`, the same kind of object the hand-built path produces. |
| `StateGraph` + `TypedDict` state | nodes are `state -> update` functions; the `add_messages` reducer appends, standing in for the message list you threaded by hand. |
| Conditional edge | `add_conditional_edges("triage", route, ...)` picks the next node from state — chapter 05's `route()` wired into the graph. |
| `interrupt_before` | pauses the run before a risky node and waits for external input — chapter 08's `require_approval`, declarative. |
| Checkpointer | `MemorySaver` (alias of `InMemorySaver`) persists state under a `thread_id`, so a paused run resumes with `invoke(None, config)` — chapter 08's `Checkpoint`. |
| The trade | free persistence, human-in-the-loop, and streaming; paid in graph boilerplate, indirection, and a moved observability seam. |

## Exercises

1. **Add a node.** The desk sometimes needs stock context on a return. Add a `check_inventory` read tool (it already exists in `shoplab.tools`) to `read_tools`, or add a fourth node that annotates the state when a returned sku is low on stock, and route to it from `triage`. Does the graph actually visit your new node on `TKT-2205`, and what did adding it cost in edges?
2. **Change the conditional-edge policy.** Right now `route` sends any `approve_refund` or `partial_refund` proposal to the gate. Tighten it: gate only when the proposed dollar amount is over some threshold (parse it from the `DECISION:` line), letting small refunds through ungated. Re-run and confirm a large refund still pauses while a small one does not — the routing policy is a design surface, same as it was in chapter 05.
3. **Resume from the checkpoint, differently.** After the interrupt fires, instead of approving, use `graph.update_state(config, ...)` to edit the proposed decision before resuming, or start a *second* `thread_id` and confirm its pause is independent of the first. What does the checkpointer let you inspect and rewrite that a plain `Checkpoint` file did not?

**Next up:** chapter 17 leaves the graph behind for a genuinely different execution model — smolagents, where the agent writes its actions as Python *code* instead of emitting JSON tool calls, the CodeAct idea you have not built by hand.